In [12]:
# Step 1 - Initialize the Spark session for the Gold analytics layer
# This notebook reads the Silver Iceberg table, computes business-ready aggregates,
# and writes the result into the Gold layer as a curated analytical table.

import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from lakehouse.spark_session import get_spark

spark = get_spark("gold-transformations")

In [13]:
# Step 2 - Read the Silver sales table from the Nessie catalog
# The Gold layer is built from the standardized Silver layer.

sales_silver = spark.table("nessie.silver.sales")
sales_silver.show(5, truncate=False)
sales_silver.printSchema()

+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------------------------------------------------+-------+--------+--------+--------+--------------+--------------------------+--------------------+--------------+---------+
|order_id      |order_date|ship_date |ship_mode     |customer_id|customer_name   |segment    |country      |city         |state     |postal_code|region|product_id     |category       |sub_category|product_name                                                  |sales  |quantity|discount|profit  |ingestion_date|ingestion_ts              |source_file         |source_system |batch_id |
+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------

In [14]:
# Step 3 - Build a business-ready Gold aggregation by category and region
# The monetary values are rounded for reporting purposes.

from pyspark.sql.functions import sum as spark_sum, round as spark_round, count

sales_gold = sales_silver.groupBy("category", "region").agg(
    spark_round(spark_sum("sales"), 2).alias("total_sales"),
    spark_round(spark_sum("profit"), 2).alias("total_profit"),
    spark_sum("quantity").alias("total_quantity"),
    count("order_id").alias("order_count")
)

sales_gold.show(20, truncate=False)
sales_gold.printSchema()

+---------------+-------+-----------+------------+--------------+-----------+
|category       |region |total_sales|total_profit|total_quantity|order_count|
+---------------+-------+-----------+------------+--------------+-----------+
|Furniture      |East   |208009.83  |3058.22     |2212          |600        |
|Technology     |East   |264973.98  |47462.04    |1942          |535        |
|Office Supplies|West   |220853.25  |52609.85    |7235          |1897       |
|Furniture      |West   |252612.74  |11504.95    |2696          |707        |
|Office Supplies|Central|167026.42  |8879.98     |5409          |1422       |
|Furniture      |Central|163797.16  |-2871.05    |1827          |481        |
|Office Supplies|East   |204922.55  |40833.62    |6455          |1710       |
|Technology     |West   |251889.5   |44289.58    |2327          |598        |
|Office Supplies|South  |125515.57  |19923.95    |3792          |993        |
|Technology     |Central|170416.31  |33697.43    |1544          

In [15]:
# Step 4 - Reset the Gold table if needed
# This is useful during development to rebuild the Gold layer cleanly.

spark.sql("DROP TABLE IF EXISTS nessie.gold.sales_by_category_region")

DataFrame[]

In [16]:
# Step 5 - Write the business-ready aggregation into the Gold layer as an Iceberg table

sales_gold.writeTo("nessie.gold.sales_by_category_region").using("iceberg").createOrReplace()   

In [18]:
# Step 6 - Query the Gold table to verify that the analytical dataset was written correctly

spark.sql("SELECT COUNT(*) FROM nessie.gold.sales_by_category_region").show()
spark.sql("SELECT * FROM nessie.gold.sales_by_category_region LIMIT 20").show(truncate=False)

+--------+
|count(1)|
+--------+
|      12|
+--------+

+---------------+-------+-----------+------------+--------------+-----------+
|category       |region |total_sales|total_profit|total_quantity|order_count|
+---------------+-------+-----------+------------+--------------+-----------+
|Furniture      |East   |208009.83  |3058.22     |2212          |600        |
|Technology     |East   |264973.98  |47462.04    |1942          |535        |
|Office Supplies|West   |220853.25  |52609.85    |7235          |1897       |
|Furniture      |West   |252612.74  |11504.95    |2696          |707        |
|Office Supplies|Central|167026.42  |8879.98     |5409          |1422       |
|Furniture      |Central|163797.16  |-2871.05    |1827          |481        |
|Office Supplies|East   |204922.55  |40833.62    |6455          |1710       |
|Technology     |West   |251889.5   |44289.58    |2327          |598        |
|Office Supplies|South  |125515.57  |19923.95    |3792          |993        |
|Technol

In [19]:
# Step 7 - Inspect the Iceberg metadata tables for the Gold table
# This confirms that the curated analytical layer is also versioned and auditable.

spark.sql("SELECT * FROM nessie.gold.sales_by_category_region.history").show(truncate=False)
spark.sql("SELECT * FROM nessie.gold.sales_by_category_region.snapshots").show(truncate=False)

+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2026-03-12 00:46:14.302|9115961248353875214|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+

+-----------------------+-------------------+---------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------